# 24030302038 — News Articles Scraper
## One War, Many Narratives

This notebook collects **news articles** about the Iran-Israel / US-linked conflict
from two sources and merges them into a single cleaned dataset.

**What this notebook does — in order:**
1. Installs required libraries
2. Collects articles via **NewsAPI** (keyword search)
3. Collects articles via **RSS feeds** (multi-source, multi-perspective)
4. **Merges** both collections, deduplicates, and saves the final dataset

**Freeze mechanism:** If an output file already exists, the scraper will skip that step and print a message. Delete the file to re-run that step.

**Output files (saved to `Scraped_Datasets/` folder):**
| File | Contents |
|---|---|
| `01_Iran_News_Raw.csv` | Raw NewsAPI articles (title, description, content) |
| `02_Iran_News_Full.csv` | NewsAPI articles with full text extracted |
| `03_Iran_News_RSS.csv` | RSS-scraped articles |
| `04_news_merge_audit.csv` | Audit trail of the merge process |
| `05_news_merged_raw.csv` | **Final merged dataset — use this for analysis** |

---
> ⚠️ **Before running:** Set your NewsAPI key in Section 2 Configuration.  
> The RSS scraper and merging require no API keys.

### The output of this scraper is taken to Datasets/News Articles as `News_Articles.csv`

>For the benefit of the scraper to work properly, please do not change any part of this scraper file

## Section 1: Install Required Libraries

In [ ]:
# Install all required libraries
%pip install -q newsapi-python newspaper3k lxml_html_clean feedparser pandas requests
print("✅ Libraries installed")

Note: you may need to restart the kernel to use updated packages.
✅ Libraries installed


## Section 2: NewsAPI Scraper

Collects articles using the [NewsAPI](https://newsapi.org/) keyword search.
- **Free tier:** last 30 days, 100 requests/day
- Outputs: `01_Iran_News_Raw.csv` → `02_Iran_News_Full.csv`


### 2.1 Configuration — Set your API key here

In [ ]:
import os
import time
import re
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
from newsapi import NewsApiClient
from newspaper import Article

# ============================================================
# ⚙️  CONFIGURATION — EDIT THIS SECTION
# ============================================================

# Your NewsAPI key — get one free at https://newsapi.org/
NEWSAPI_KEY = "2a1bb46f09f247a68e51aef37d724da3"

# Output folder (relative to this notebook)
OUTPUT_DIR = Path(r'D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\News Articles')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keywords to search (Iran-Israel conflict)
KEYWORDS = [
    "Iran Israel war",
    "Iran Israel India",
    "Middle East conflict India",
    "Iran war oil India",
    "Iran Israel ceasefire",
]

# Date range (free tier: last 28 days only)
END_DATE   = datetime.today().strftime("%Y-%m-%d")
START_DATE = (datetime.today() - timedelta(days=28)).strftime("%Y-%m-%d")

# Output file paths
RAW_PATH  = OUTPUT_DIR / "01_Iran_News_Raw.csv"
FULL_PATH = OUTPUT_DIR / "02_Iran_News_Full.csv"

# ============================================================
print(f"Output folder : {OUTPUT_DIR.resolve()}")
print(f"Date range    : {START_DATE} to {END_DATE}")
print(f"Keywords      : {len(KEYWORDS)}")


Output folder : D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\News Articles
Date range    : 2026-03-07 to 2026-04-04
Keywords      : 5


### 2.2 Collect Raw Articles from NewsAPI

In [ ]:
if RAW_PATH.exists():
    df_raw = pd.read_csv(RAW_PATH)
    print(f"✅ Already collected — {len(df_raw)} articles found in {RAW_PATH.name}")
    print("   Delete the file to re-collect.")
else:
    newsapi = NewsApiClient(api_key=NEWSAPI_KEY)
    all_articles = []

    for keyword in KEYWORDS:
        response = newsapi.get_everything(
            q=keyword,
            language="en",
            from_param=START_DATE,
            to=END_DATE,
            sort_by="relevancy",
            page_size=100,
        )
        if response["status"] == "ok":
            for article in response["articles"]:
                all_articles.append({
                    "source":       article["source"]["name"],
                    "title":        article["title"],
                    "description":  article["description"],
                    "content":      article["content"],
                    "published_at": article["publishedAt"],
                    "keyword_used": keyword,
                    "url":          article["url"],
                })
            print(f"  '{keyword}' → {len(response['articles'])} articles")

    df_raw = (
        pd.DataFrame(all_articles)
        .drop_duplicates(subset=["title"])
        .reset_index(drop=True)
    )
    df_raw.to_csv(RAW_PATH, index=False)
    print(f"\n✅ Collected {len(df_raw)} articles → saved to {RAW_PATH.name}")

print(f"\nSource distribution:")
print(df_raw["source"].value_counts().to_string())


✅ Already collected — 388 articles found in 01_Iran_News_Raw.csv
   Delete the file to re-collect.

Source distribution:
source
Al Jazeera English              122
The Indian Express               47
Business Insider                 28
NPR                              27
RT                               26
BBC News                         18
Yahoo Entertainment              12
CBC News                          9
The Times of India                9
Vox                               8
Abcnews.com                       7
Wired                             7
Skift                             6
Gizmodo.com                       6
Democracy Now!                    5
The New Yorker                    4
CoinDesk                          4
The New Republic                  4
The Atlantic                      3
The American Conservative         2
The Verge                         2
Kotaku                            2
Naturalnews.com                   2
Daily Beast                       2
The Inte

### 2.3 Extract Full Text from Article URLs

In [ ]:
def get_full_text(url):
    """Extract full article text via newspaper3k. Returns None if extraction fails."""
    try:
        article = Article(url)
        article.download()
        article.parse()
        if article.text and len(article.text) > 100:
            return article.text
        return None
    except Exception:
        return None


if FULL_PATH.exists():
    df_full = pd.read_csv(FULL_PATH)
    print(f"✅ Already extracted — {len(df_full)} articles found in {FULL_PATH.name}")
    print("   Delete the file to re-extract.")
else:
    df_raw = pd.read_csv(RAW_PATH)
    print(f"Extracting full text for {len(df_raw)} articles...")
    print("This will take ~10–15 minutes (1 second per article)\n")

    full_texts = []
    for i, url in enumerate(df_raw["url"]):
        text = get_full_text(url)
        full_texts.append(text)
        status = f"✅ {len(text):,} chars" if text else "❌ Failed"
        print(f"  {i+1}/{len(df_raw)} — {df_raw['source'].iloc[i]:<28} {status}")
        time.sleep(1)

    df_raw["full_text"] = full_texts
    # Fallback: use title + description + content when newspaper3k fails
    df_raw["full_text"] = df_raw.apply(
        lambda row: row["full_text"] if row["full_text"]
        else " ".join(filter(None, [
            str(row.get("title", "") or ""),
            str(row.get("description", "") or ""),
            str(row.get("content", "") or ""),
        ])),
        axis=1,
    )
    df_full = df_raw.copy()
    df_full.to_csv(FULL_PATH, index=False)

    success = sum(1 for t in full_texts if t)
    print(f"\n✅ Full text extracted: {success} full | {len(df_raw)-success} fallback")
    print(f"   Saved to {FULL_PATH.name}")

print(f"\nAvg text length: {df_full['full_text'].str.len().mean():.0f} chars")


✅ Already extracted — 388 articles found in 02_Iran_News_Full.csv
   Delete the file to re-extract.

Avg text length: 4516 chars


## Section 3: RSS Feed Scraper

Collects articles from curated RSS feeds across multiple outlets and perspectives.
- No API key required
- Covers: Western, Arab/ME, South Asian, Russian, Turkish, Israeli perspectives
- Output: `03_Iran_News_RSS.csv`


### 3.1 RSS Imports and Setup

In [ ]:
import html
from datetime import timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlsplit, urlunsplit

import feedparser
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RSS_PATH = OUTPUT_DIR / "03_Iran_News_RSS.csv"

# ── Scraper settings ──────────────────────────────────────────
TOPIC_LABEL           = "Iran-US conflict"
LOOKBACK_DAYS         = 60       # How far back to look
MAX_ARTICLES_PER_SOURCE = None   # None = no cap per source
SCAN_LIMIT_PER_FEED   = None     # None = scan all entries
REQUEST_DELAY_SECONDS = 0.5
MIN_FULL_TEXT_CHARS   = 120
RELEVANCE_MODE        = "broad"  # "broad" or "strict"


### 3.2 Source Catalog — All RSS Feeds

In [ ]:
# ============================================================
# SOURCE CATALOG — Add or remove sources here
# Each source needs: source name, perspective, country, feed_urls list
# ============================================================
SOURCES = [
    {
        "source": "Reuters", "perspective": "Western", "country_or_region": "Global",
        "feed_urls": [
            {"feed_name": "World News", "feed_url": "https://feeds.reuters.com/Reuters/worldNews"},
            {"feed_name": "Top News",   "feed_url": "https://feeds.reuters.com/reuters/topNews"},
        ],
    },
    {
        "source": "BBC News", "perspective": "Western", "country_or_region": "United Kingdom",
        "feed_urls": [
            {"feed_name": "World",       "feed_url": "https://feeds.bbci.co.uk/news/world/rss.xml"},
            {"feed_name": "Middle East", "feed_url": "https://feeds.bbci.co.uk/news/world/middle_east/rss.xml"},
        ],
    },
    {
        "source": "The Guardian", "perspective": "Western", "country_or_region": "United Kingdom",
        "feed_urls": [
            {"feed_name": "World",       "feed_url": "https://www.theguardian.com/world/rss"},
            {"feed_name": "Middle East", "feed_url": "https://www.theguardian.com/world/middleeast/rss"},
        ],
    },
    {
        "source": "NPR", "perspective": "Western", "country_or_region": "United States",
        "feed_urls": [
            {"feed_name": "World", "feed_url": "https://feeds.npr.org/1004/rss.xml"},
        ],
    },
    {
        "source": "CBS News", "perspective": "Western", "country_or_region": "United States",
        "feed_urls": [
            {"feed_name": "World",       "feed_url": "https://www.cbsnews.com/latest/rss/world"},
            {"feed_name": "Top Stories", "feed_url": "https://www.cbsnews.com/latest/rss/main"},
        ],
    },
    {
        "source": "Al Jazeera", "perspective": "Arab / Middle East", "country_or_region": "Qatar",
        "feed_urls": [
            {"feed_name": "All News", "feed_url": "https://www.aljazeera.com/xml/rss/all.xml"},
        ],
    },
    {
        "source": "Middle East Eye", "perspective": "Arab / Middle East",
        "country_or_region": "United Kingdom / Middle East",
        "feed_urls": [
            {"feed_name": "Main RSS", "feed_url": "https://www.middleeasteye.net/rss"},
        ],
    },
    {
        "source": "Arab News", "perspective": "Arab / Middle East", "country_or_region": "Saudi Arabia",
        "feed_urls": [
            {"feed_name": "Main RSS", "feed_url": "https://www.arabnews.com/rss.xml"},
        ],
    },
    {
        "source": "Jerusalem Post", "perspective": "Israeli", "country_or_region": "Israel",
        "feed_urls": [
            {"feed_name": "Headlines", "feed_url": "https://www.jpost.com/rss/rssfeedsheadlines.aspx"},
        ],
    },
    {
        "source": "Times of Israel", "perspective": "Israeli", "country_or_region": "Israel",
        "feed_urls": [
            {"feed_name": "Main Feed", "feed_url": "https://www.timesofisrael.com/feed/"},
        ],
    },
    {
        "source": "RT", "perspective": "Russian", "country_or_region": "Russia",
        "feed_urls": [
            {"feed_name": "Main RSS", "feed_url": "https://www.rt.com/rss/"},
        ],
    },
    {
        "source": "TASS", "perspective": "Russian", "country_or_region": "Russia",
        "feed_urls": [
            {"feed_name": "Main RSS", "feed_url": "https://tass.com/rss/v2.xml"},
        ],
    },
    {
        "source": "The Hindu", "perspective": "South Asian", "country_or_region": "India",
        "feed_urls": [
            {"feed_name": "International", "feed_url": "https://www.thehindu.com/news/international/feeder/default.rss"},
        ],
    },
    {
        "source": "NDTV", "perspective": "South Asian", "country_or_region": "India",
        "feed_urls": [
            {"feed_name": "World News", "feed_url": "https://feeds.feedburner.com/ndtvnews-world-news"},
        ],
    },
    {
        "source": "Dawn", "perspective": "South Asian", "country_or_region": "Pakistan",
        "feed_urls": [
            {"feed_name": "Home", "feed_url": "https://www.dawn.com/feeds/home"},
        ],
    },
    {
        "source": "Hindustan Times", "perspective": "South Asian", "country_or_region": "India",
        "feed_urls": [
            {"feed_name": "World News",  "feed_url": "https://www.hindustantimes.com/feeds/rss/world-news/rssfeed.xml"},
            {"feed_name": "Geopolitics", "feed_url": "https://www.hindustantimes.com/feeds/rss/geopolitics/rssfeed.xml"},
        ],
    },
    {
        "source": "The Indian Express", "perspective": "South Asian", "country_or_region": "India",
        "feed_urls": [
            {"feed_name": "World",            "feed_url": "https://indianexpress.com/section/world/feed/"},
            {"feed_name": "Explained Global", "feed_url": "https://indianexpress.com/section/explained/explained-global/feed/"},
        ],
    },
    {
        "source": "TRT World", "perspective": "Turkish", "country_or_region": "Turkey",
        "feed_urls": [
            {"feed_name": "Main RSS", "feed_url": "https://www.trtworld.com/rss"},
        ],
    },
    {
        "source": "Daily Sabah", "perspective": "Turkish", "country_or_region": "Turkey",
        "feed_urls": [
            {"feed_name": "World",    "feed_url": "https://www.dailysabah.com/rss/world"},
            {"feed_name": "Mid-East", "feed_url": "https://www.dailysabah.com/rss/world/mid-east"},
        ],
    },
]

print(f"✅ Source catalog loaded: {len(SOURCES)} sources")
perspective_count = {}
for s in SOURCES:
    p = s["perspective"]
    perspective_count[p] = perspective_count.get(p, 0) + 1
for p, c in sorted(perspective_count.items()):
    print(f"   {p:<25} — {c} source(s)")


✅ Source catalog loaded: 19 sources
   Arab / Middle East        — 3 source(s)
   Israeli                   — 2 source(s)
   Russian                   — 2 source(s)
   South Asian               — 5 source(s)
   Turkish                   — 2 source(s)
   Western                   — 5 source(s)


### 3.3 Helper Functions

In [ ]:
# ── Relevance patterns ────────────────────────────────────────
TERM_GROUPS_RAW = {
    "iran": [
        ("Iran",               r"\biran\b"),
        ("Iranian",            r"\biranian\b"),
        ("Tehran",             r"\btehran\b"),
        ("IRGC",               r"\birgc\b"),
        ("Revolutionary Guard",r"revolutionary guard"),
        ("Khamenei",           r"\bkhamenei\b"),
        ("Hormuz",             r"\bhormuz\b"),
        ("Persian Gulf",       r"\bpersian gulf\b"),
    ],
    "us": [
        ("United States", r"\bunited states\b"),
        ("US",            r"\bUS\b"),
        ("America",       r"\bamerica\b"),
        ("Washington",    r"\bwashington\b"),
        ("Pentagon",      r"\bpentagon\b"),
        ("CENTCOM",       r"\bcentcom\b"),
    ],
    "conflict": [
        ("war",       r"\bwar\b"),
        ("strike",    r"\bstrike[s]?\b"),
        ("airstrike", r"\bairstrike[s]?\b"),
        ("missile",   r"\bmissile[s]?\b"),
        ("attack",    r"\battack[s]?\b"),
        ("ceasefire", r"\bceasefire\b"),
    ],
    "regional": [
        ("Israel",      r"\bisrael\b"),
        ("Middle East", r"\bmiddle east\b"),
        ("Gulf",        r"\bgulf\b"),
        ("Hezbollah",   r"\bhezbollah\b"),
        ("Lebanon",     r"\blebanon\b"),
    ],
}
COMPILED_GROUPS = {
    group: [(label, re.compile(pattern, re.IGNORECASE)) for label, pattern in patterns]
    for group, patterns in TERM_GROUPS_RAW.items()
}


# ── Text cleaning ─────────────────────────────────────────────
def rss_clean_text(text):
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def rss_normalize_url(url):
    parsed = urlsplit(url or "")
    path = parsed.path.rstrip("/")
    return urlunsplit((parsed.scheme.lower(), parsed.netloc.lower(), path, "", ""))


def rss_title_key(title):
    title = re.sub(r"[^a-z0-9]+", " ", (title or "").lower())
    return re.sub(r"\s+", " ", title).strip()


# ── Date parsing ──────────────────────────────────────────────
def rss_parse_date(entry):
    for attr in ["published_parsed", "updated_parsed", "created_parsed"]:
        value = getattr(entry, attr, None) or entry.get(attr)
        if value:
            try:
                return datetime(*value[:6], tzinfo=timezone.utc)
            except Exception:
                pass
    for attr in ["published", "updated", "created"]:
        value = getattr(entry, attr, None) or entry.get(attr)
        if value:
            try:
                parsed = parsedate_to_datetime(value)
                if parsed.tzinfo is None:
                    parsed = parsed.replace(tzinfo=timezone.utc)
                return parsed.astimezone(timezone.utc)
            except Exception:
                pass
    return datetime.now(timezone.utc)


# ── HTTP session ──────────────────────────────────────────────
def build_http_session():
    session = requests.Session()
    retry = Retry(total=2, backoff_factor=0.5,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET", "HEAD"])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/rss+xml, application/xml, text/xml, */*;q=0.8",
    })
    return session

HTTP_SESSION = build_http_session()


def fetch_feed(feed_url, timeout=25):
    try:
        response = HTTP_SESSION.get(feed_url, timeout=timeout, allow_redirects=True)
        response.raise_for_status()
        parsed = feedparser.parse(response.content)
        if parsed.entries:
            return parsed, response.url, None
    except Exception as exc:
        pass
    parsed = feedparser.parse(feed_url)
    warning = None if parsed.entries else "No entries returned"
    return parsed, feed_url, warning


# ── Relevance checks ──────────────────────────────────────────
def detect_matches(text):
    cleaned = rss_clean_text(text)
    matched = {}
    for group, patterns in COMPILED_GROUPS.items():
        hits = [label for label, pattern in patterns if pattern.search(cleaned)]
        if hits:
            matched[group] = sorted(set(hits))
    score = sum(len(v) for v in matched.values())
    return matched, score


def passes_filter(text, mode=RELEVANCE_MODE):
    matched, score = detect_matches(text)
    has_iran     = "iran"     in matched
    has_us       = "us"       in matched
    has_conflict = "conflict" in matched
    has_regional = "regional" in matched
    if mode == "strict":
        return has_iran and (has_us or (has_conflict and score >= 3))
    return has_iran and (has_us or has_conflict or (has_regional and score >= 2) or score >= 4)


# ── Full text extraction ──────────────────────────────────────
def rss_extract_full_text(url):
    try:
        article = Article(url=url, language="en",
                          browser_user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
        article.download()
        article.parse()
        text = rss_clean_text(article.text)
        if len(text) >= MIN_FULL_TEXT_CHARS:
            return text
    except Exception:
        pass
    return None


print("✅ Helper functions defined")


✅ Helper functions defined


### 3.4 Run the RSS Scraper

In [ ]:
if RSS_PATH.exists():
    articles_df = pd.read_csv(RSS_PATH)
    print(f"✅ Already scraped — {len(articles_df)} articles found in {RSS_PATH.name}")
    print("   Delete the file to re-scrape.")
else:
    cutoff      = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)
    records     = []
    seen_urls   = set()
    seen_titles = set()

    for source in SOURCES:
        print("=" * 80)
        print(f"Source:      {source['source']}  |  Perspective: {source['perspective']}")
        kept_for_source = 0

        for feed_config in source["feed_urls"]:
            print(f"  Feed: {feed_config['feed_name']}  ({feed_config['feed_url']})")
            feed, resolved_url, warning = fetch_feed(feed_config["feed_url"])
            if warning:
                print(f"  ⚠️  Warning: {warning}")
            if not getattr(feed, "entries", None):
                print("  Collected: 0")
                continue

            entries = feed.entries
            if SCAN_LIMIT_PER_FEED:
                entries = entries[:SCAN_LIMIT_PER_FEED]

            kept_for_feed = 0
            for entry in entries:
                pub_date     = rss_parse_date(entry)
                if pub_date < cutoff:
                    continue
                url          = getattr(entry, "link", "") or entry.get("id", "")
                if not url:
                    continue
                norm_url     = rss_normalize_url(url)
                title        = rss_clean_text(getattr(entry, "title", ""))
                summary      = rss_clean_text(getattr(entry, "summary", "") or
                                              getattr(entry, "description", ""))
                preview      = rss_clean_text(f"{title}. {summary}")
                source_title = (source["source"], rss_title_key(title))

                if not title:
                    continue
                keep, score, matched = detect_matches(preview)
                if not passes_filter(preview):
                    continue
                if norm_url in seen_urls or source_title in seen_titles:
                    continue

                full_text = rss_extract_full_text(url)
                if REQUEST_DELAY_SECONDS > 0:
                    time.sleep(REQUEST_DELAY_SECONDS)

                used_fallback = False
                if not full_text:
                    full_text    = preview
                    used_fallback = True

                combined = rss_clean_text(f"{preview}. {full_text}")
                if not passes_filter(combined):
                    continue

                seen_urls.add(norm_url)
                seen_titles.add(source_title)
                matched_terms = sorted({t for terms in matched.values() for t in terms})

                records.append({
                    "topic":             TOPIC_LABEL,
                    "source":            source["source"],
                    "perspective":       source["perspective"],
                    "country_or_region": source["country_or_region"],
                    "feed_name":         feed_config["feed_name"],
                    "feed_url":          resolved_url,
                    "published_at":      pub_date.isoformat().replace("+00:00", "Z"),
                    "title":             title,
                    "summary":           summary,
                    "full_text":         full_text,
                    "url":               url,
                    "domain":            urlsplit(url).netloc.lower(),
                    "relevance_score":   score,
                    "matched_groups":    ", ".join(sorted(matched)),
                    "matched_terms":     ", ".join(matched_terms),
                    "used_fallback_text": used_fallback,
                    "full_text_chars":   len(full_text),
                    "collection_method": f"rss + newspaper3k ({RELEVANCE_MODE})",
                })
                kept_for_feed   += 1
                kept_for_source += 1
                if MAX_ARTICLES_PER_SOURCE and kept_for_source >= MAX_ARTICLES_PER_SOURCE:
                    break

            print(f"  Collected: {kept_for_feed} articles")
            if MAX_ARTICLES_PER_SOURCE and kept_for_source >= MAX_ARTICLES_PER_SOURCE:
                break

        print(f"Source total: {kept_for_source}")

    articles_df = (
        pd.DataFrame(records)
        .sort_values(["published_at", "source"], ascending=[False, True])
        .reset_index(drop=True)
    ) if records else pd.DataFrame()

    articles_df.to_csv(RSS_PATH, index=False, encoding="utf-8-sig")
    print(f"\n✅ RSS scraping complete — {len(articles_df)} articles → {RSS_PATH.name}")

# Coverage summary
print(f"\n{'='*55}")
print("RSS COVERAGE SUMMARY")
print(f"{'='*55}")
print(f"Total articles : {len(articles_df)}")
if not articles_df.empty:
    print(f"Date range     : {articles_df['published_at'].min()[:10]} to {articles_df['published_at'].max()[:10]}")
    print(f"\nBy perspective:")
    print(articles_df["perspective"].value_counts().to_string())
    print(f"\nBy source:")
    print(articles_df["source"].value_counts().to_string())


✅ Already scraped — 400 articles found in 03_Iran_News_RSS.csv
   Delete the file to re-scrape.

RSS COVERAGE SUMMARY
Total articles : 400
Date range     : 2026-02-04 to 2026-03-23

By perspective:
perspective
South Asian           230
Russian                73
Turkish                40
Arab / Middle East     29
Western                24
Israeli                 4

By source:
source
The Indian Express    144
Daily Sabah            40
TASS                   39
Hindustan Times        35
RT                     34
The Hindu              26
The Guardian           21
Middle East Eye        16
Dawn                   13
Al Jazeera             13
NDTV                   12
Times of Israel         4
Sky News                3


## Section 4: Merge Both Datasets

Combines the NewsAPI articles (`02_Iran_News_Full.csv`) and the RSS articles
(`03_Iran_News_RSS.csv`) into one clean corpus.

**Steps performed:**
1. Load both files
2. Filter NewsAPI to credible sources, add perspective labels
3. Filter RSS to quality threshold (relevance ≥ 5, text ≥ 200 chars)
4. Concatenate both
5. Deduplicate by normalized URL, then by title+date
6. Save audit trail and final merged file

**Outputs:**
- `04_news_merge_audit.csv` — row-by-row audit of merge decisions
- `05_news_merged_raw.csv` — **final merged dataset for analysis**


### 4.1 Merge Configuration

In [ ]:
import json
import numpy as np
from urllib.parse import urlsplit, urlunsplit

AUDIT_PATH  = OUTPUT_DIR / "04_news_merge_audit.csv"
MERGED_PATH = OUTPUT_DIR / "05_news_merged_raw.csv"

# NewsAPI credible sources to keep
OLD_CREDIBLE_SOURCES = [
    "Al Jazeera English", "The Indian Express", "Business Insider", "NPR",
    "RT", "BBC News", "Yahoo Entertainment", "CBC News", "The Times of India",
    "Vox", "Abcnews.com", "Democracy Now!", "The New Yorker", "The Atlantic",
    "The Intercept", "CBS News", "DW (English)", "Newsweek",
    "The Jerusalem Post", "Forbes",
]

# Perspective and country labels for NewsAPI sources
SOURCE_PERSPECTIVE_MAP = {
    "Al Jazeera English": "Arab / Middle East",
    "The Indian Express": "South Asian",
    "Business Insider":   "Western",
    "NPR":                "Western",
    "RT":                 "Russian",
    "BBC News":           "Western",
    "Yahoo Entertainment":"Western",
    "CBC News":           "Western",
    "The Times of India": "South Asian",
    "Vox":                "Western",
    "Abcnews.com":        "Western",
    "Democracy Now!":     "Western",
    "The New Yorker":     "Western",
    "The Atlantic":       "Western",
    "The Intercept":      "Western",
    "CBS News":           "Western",
    "DW (English)":       "Western",
    "Newsweek":           "Western",
    "The Jerusalem Post": "Israeli",
    "Forbes":             "Western",
}

SOURCE_COUNTRY_MAP = {
    "Al Jazeera English": "Qatar",
    "The Indian Express": "India",
    "Business Insider":   "United States",
    "NPR":                "United States",
    "RT":                 "Russia",
    "BBC News":           "United Kingdom",
    "Yahoo Entertainment":"United States",
    "CBC News":           "Canada",
    "The Times of India": "India",
    "Vox":                "United States",
    "Abcnews.com":        "United States",
    "Democracy Now!":     "United States",
    "The New Yorker":     "United States",
    "The Atlantic":       "United States",
    "The Intercept":      "United States",
    "CBS News":           "United States",
    "DW (English)":       "Germany",
    "Newsweek":           "United States",
    "The Jerusalem Post": "Israel",
    "Forbes":             "United States",
}

SOURCE_NORMALIZATION = {
    "Al Jazeera":          "Al Jazeera English",
    "Indian Express":      "The Indian Express",
    "Jerusalem Post":      "The Jerusalem Post",
}

def normalize_source(name):
    return SOURCE_NORMALIZATION.get(str(name).strip(), str(name).strip())

def merge_normalize_url(url):
    parsed = urlsplit(str(url or ""))
    return urlunsplit((parsed.scheme.lower(), parsed.netloc.lower(),
                       parsed.path.rstrip("/"), "", ""))

def normalize_title(title):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ",
                  str(title or "").lower())).strip()

def title_date_key(title, published_at):
    t_key = normalize_title(title)[:60]
    try:
        d_key = str(published_at)[:10]
    except Exception:
        d_key = ""
    return f"{t_key}||{d_key}"

def parse_date_safe(val):
    try:
        return pd.to_datetime(val, utc=True)
    except Exception:
        return pd.NaT

print("✅ Merge configuration ready")


✅ Merge configuration ready


### 4.2 Build Merged Corpus

In [ ]:
if MERGED_PATH.exists() and AUDIT_PATH.exists():
    news_merged = pd.read_csv(MERGED_PATH)
    audit_df    = pd.read_csv(AUDIT_PATH)
    print(f"✅ Already merged — {len(news_merged)} articles found in {MERGED_PATH.name}")
    print("   Delete the file to re-merge.")
else:
    audit_rows = []

    # ── Load and prepare NewsAPI file ────────────────────────
    old_df = pd.read_csv(OUTPUT_DIR / "02_Iran_News_Full.csv")
    audit_rows.append({"stage": "newsapi_raw_loaded",
                       "row_count": len(old_df),
                       "note": "Original 02_Iran_News_Full.csv rows"})

    old_df = old_df[old_df["source"].isin(OLD_CREDIBLE_SOURCES)].copy()
    audit_rows.append({"stage": "newsapi_credible_filtered",
                       "row_count": len(old_df),
                       "note": "After filtering to credible sources"})

    old_df["source"]           = old_df["source"].apply(normalize_source)
    old_df["perspective"]      = old_df["source"].map(SOURCE_PERSPECTIVE_MAP).fillna("Unknown")
    old_df["country_or_region"]= old_df["source"].map(SOURCE_COUNTRY_MAP).fillna("Unknown")
    old_df["relevance_score"]  = 10
    old_df["dataset_source"]   = "NewsAPI"
    old_df["collection_method"]= "NewsAPI + newspaper3k"
    old_df["full_text_chars"]  = old_df["full_text"].str.len().fillna(0).astype(int)
    old_df["extraction_status"]= "newsapi_extracted"

    old_slim = old_df[[
        "source", "perspective", "country_or_region", "published_at",
        "title", "full_text", "url", "relevance_score", "dataset_source",
        "collection_method", "extraction_status", "full_text_chars",
    ]].copy()

    # ── Load and prepare RSS file ─────────────────────────────
    new_df = pd.read_csv(OUTPUT_DIR / "03_Iran_News_RSS.csv")
    audit_rows.append({"stage": "rss_raw_loaded",
                       "row_count": len(new_df),
                       "note": "Original 03_Iran_News_RSS.csv rows"})

    new_df = new_df[
        (new_df["relevance_score"] >= 5) &
        (new_df["full_text_chars"] >= 200)
    ].copy()
    audit_rows.append({"stage": "rss_quality_filtered",
                       "row_count": len(new_df),
                       "note": "relevance_score ≥ 5 and full_text_chars ≥ 200"})

    new_df["source"]           = new_df["source"].apply(normalize_source)
    new_df["dataset_source"]   = "RSS_Scraper"
    new_df["extraction_status"]= np.where(
        new_df["used_fallback_text"].fillna(False), "rss_fallback", "rss_full_text")

    new_slim = new_df[[
        "source", "perspective", "country_or_region", "published_at",
        "title", "full_text", "url", "relevance_score", "dataset_source",
        "collection_method", "extraction_status", "full_text_chars",
    ]].copy()

    # ── Concatenate ───────────────────────────────────────────
    merged = pd.concat([old_slim, new_slim], ignore_index=True)
    audit_rows.append({"stage": "concatenated",
                       "row_count": len(merged),
                       "note": "NewsAPI + RSS concatenated"})

    # ── Add dedup columns ─────────────────────────────────────
    merged["normalized_url"]   = merged["url"].apply(merge_normalize_url)
    merged["normalized_title"] = merged["title"].apply(normalize_title)
    merged["published_ts"]     = merged["published_at"].apply(parse_date_safe)
    merged["published_date"]   = merged["published_ts"].dt.date.astype(str)
    merged["title_date_key"]   = merged.apply(
        lambda r: title_date_key(r["title"], r["published_at"]), axis=1)

    # ── Deduplicate by URL ────────────────────────────────────
    before = len(merged)
    merged = merged.sort_values(["relevance_score", "full_text_chars"],
                                ascending=[False, False])
    has_url = merged["normalized_url"].str.strip().ne("")
    merged  = pd.concat([
        merged.loc[has_url].drop_duplicates(subset=["normalized_url"], keep="first"),
        merged.loc[~has_url],
    ], ignore_index=True)
    audit_rows.append({"stage": "dedup_by_url",
                       "row_count": len(merged),
                       "note": f"Removed {before - len(merged)} duplicate URLs"})

    # ── Deduplicate by title + date ───────────────────────────
    before = len(merged)
    merged = merged.drop_duplicates(subset=["title_date_key"], keep="first").reset_index(drop=True)
    audit_rows.append({"stage": "dedup_by_title_date",
                       "row_count": len(merged),
                       "note": f"Removed {before - len(merged)} duplicate title+date pairs"})

    news_merged = merged.copy()
    audit_df    = pd.DataFrame(audit_rows)

    audit_df.to_csv(AUDIT_PATH, index=False)
    news_merged.to_csv(MERGED_PATH, index=False)
    print(f"✅ Merge complete → {len(news_merged)} articles saved to {MERGED_PATH.name}")
    print(f"✅ Audit trail saved to {AUDIT_PATH.name}")

print(f"\n{'='*55}")
print("MERGE AUDIT TRAIL")
print(f"{'='*55}")
print(audit_df.to_string(index=False))


✅ Already merged — 710 articles found in 05_news_merged_raw.csv
   Delete the file to re-merge.

MERGE AUDIT TRAIL
                          stage  row_count                                                              note
               file2_raw_loaded        388                      Original File 2 (02_Iran_News_Full.csv) rows
file2_filtered_credible_sources        332                                  Credible-source-only File 2 rows
    file2_reextraction_complete        332                                {"fallback_network_disabled": 332}
               file3_raw_loaded        400 Original File 3 (iran_us_news_20260323_200825.csv) RSS/Codex rows
         file3_filtered_quality        382  relevance_score >= 5, full_text_chars >= 200, source != Sky News
   file2_and_file3_concatenated        714                            File 2 and File 3 corpora concatenated
           dedup_normalized_url        710                                  Removed 4 rows by normalized URL
             

### 4.3 Final Dataset Summary

In [ ]:
df = pd.read_csv(MERGED_PATH)

print("=" * 60)
print("FINAL MERGED DATASET SUMMARY")
print("=" * 60)
print(f"Total articles   : {len(df)}")
print(f"Unique sources   : {df['source'].nunique()}")
print(f"Unique perspectives: {df['perspective'].nunique()}")
print(f"Date range       : {df['published_date'].min()} to {df['published_date'].max()}")
print(f"Avg text length  : {df['full_text'].str.len().mean():.0f} chars")

print(f"\nDataset source mix:")
print(df["dataset_source"].value_counts().to_string())

print(f"\nPerspective distribution:")
print(df["perspective"].value_counts().to_string())

print(f"\nTop sources:")
print(df["source"].value_counts().head(15).to_string())

print(f"\n{'='*60}")
print("OUTPUT FILES SUMMARY")
print(f"{'='*60}")
for fname, desc in [
    ("01_Iran_News_Raw.csv",     "NewsAPI raw articles"),
    ("02_Iran_News_Full.csv",    "NewsAPI with full text"),
    ("03_Iran_News_RSS.csv",     "RSS-scraped articles"),
    ("04_news_merge_audit.csv",  "Merge audit trail"),
    ("05_news_merged_raw.csv",   "✅ FINAL — use this for analysis"),
]:
    path = OUTPUT_DIR / fname
    if path.exists():
        size_kb = path.stat().st_size // 1024
        print(f"  ✅ {fname:<35} {size_kb:>5} KB  — {desc}")
    else:
        print(f"  ❌ {fname:<35}  MISSING")


FINAL MERGED DATASET SUMMARY
Total articles   : 710
Unique sources   : 29
Unique perspectives: 6
Date range       : 2026-02-04 to 2026-03-23
Avg text length  : 2988 chars

Dataset source mix:
dataset_source
File3_RSS_Codex    382
File2_NewsAPI      328

Perspective distribution:
perspective
South Asian           280
Western               145
Arab / Middle East    145
Russian                95
Turkish                40
Israeli                 5

Top sources:
source
The Indian Express     186
Al Jazeera English     131
RT                      60
Daily Sabah             40
Hindustan Times         35
TASS                    35
Business Insider        28
NPR                     27
The Hindu               25
The Guardian            20
BBC News                18
Middle East Eye         14
Dawn                    13
NDTV                    12
Yahoo Entertainment     10

OUTPUT FILES SUMMARY
  ✅ 01_Iran_News_Raw.csv                  226 KB  — NewsAPI raw articles
  ✅ 02_Iran_News_Full.csv      